In the previous IBM notebooks, Collaborative Filtering was done with KNN. KNN looks at a user, finds the 5 users who have similar rating histories, and averages their ratings for the target movie.

**Matrix Factorization** approaches the problem completely differently. It asks: "What if we could represent every user and every movie as a small vector of numbers (an embedding), and predict the rating by taking their dot product?"

Predicted Rating$_{u,i} = v_{user} * v_{item}$

<h2>The "Cold Start" Problem</h2>

In common PhD literature, you will constantly see this scenario when trying to recommend brand new movies.

<h3>Collaborative Filtering</h3> 
(whether User-Based, Item-Based, or SVD) is entirely blind to metadata. It cannot read the title, and it does not know the genre. Its entire universe is bounded by the `user-item` interaction matrix.If a movie has zero ratings, its column in that mathematical matrix is completely blank (or filled with zeros/nulls).
<br>

**Deconstructing Item-Based vs. User-Based CF**
To understand why Item-Based CF fails here, we have to look at exactly how both memory-based approaches calculate "similarity."

**1. User-Based CF** 
The Goal: Find similar users.The Math: It compares rows in the matrix. User $A$ and User $B$ are similar if they gave similar ratings to the exact same overlapping set of movies.**The Recommendation:** If User $A$ is similar to User $B$, recommend movies User $B$ liked that User $A$ hasn't seen.

**2. Item-Based CF** (The IBM Module's other method)The Goal: Find similar items.The Math: It compares columns in the matrix. Item $X$ and Item $Y$ are similar if they were rated similarly by the exact same overlapping group of users.The Recommendation: If you liked Item $X$, the system recommends Item $Y$, because the crowd of users who liked $X$ also tended to like $Y$.

**Why the New Movie Fails:** Look closely at the math for Item-Based CF. Two items are only similar if they share a common audience of users.If our brand-new movie has zero ratings, it shares zero users with any other movie in the database. When the algorithm tries to calculate the cosine similarity between the new movie's blank vector and "The Matrix's" highly populated vector, the mathematical result is $0$. The model literally cannot see the new movie to compare it to anything.This is the golden rule of Collaborative Filtering: Similarity is defined exclusively by shared human interaction. No interactions, no similarity.

I shall use the Python library called Surprise (pip install scikit-surprise). It is built specifically for Recommender Systems and makes implementing MF algorithms (like SVD) very clean.

First Task:
Before I can train any model, I have to prepare the data. The Surprise library doesn't take raw pandas DataFrames directly. It requires the data to be in a very specific format: a structure containing only three columns: `userID`, `itemID`, and `rating`.

In [1]:
from surprise import Dataset, SVD, Reader
from surprise.accuracy import rmse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Loading the datasets

movie_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/BxZuF3FrO7Bdw6McwsBaBw/movies.csv')
rating_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/R-bYYyyf7s3IUE5rsssmMw/ratings.csv')
tag_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/UZKHhXSl7Ft7t9mfUFZJPQ/tags.csv')

In [3]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [4]:
rating_df.isna().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [5]:
rating_df.drop(columns='timestamp', inplace=True)

In [6]:
rating_df.rename(columns={'userId':'userID', 'movieId':'itemID'}, inplace=True)

In [7]:
rating_df.head()

,userID,itemID,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


In [8]:
# A reader is still needed but only the rating_scale param is required.
min_r, max_r = rating_df.rating.min(),rating_df.rating.max() 

reader = Reader(rating_scale=(min_r, max_r))

In [9]:
# The columns must correspond to user id, item id and ratings (in that order).
data = Dataset.load_from_df(rating_df[["userID", "itemID", "rating"]], reader)

In [10]:
type(data)

surprise.dataset.DatasetAutoFolds

**Splitting the dataset into train, test, split**

In [11]:
from surprise.model_selection import train_test_split

train_set, test_set = train_test_split(data, test_size=0.25, random_state=42)

In [12]:
# let's see what type of objects are the train and test sets:

print(type(train_set))
print(type(test_set))

<class 'surprise.trainset.Trainset'>
<class 'list'>


In [13]:
# The test set is a list of tuples, 

test_set[:5]

[(50, 4282, 3.5),
 (603, 2993, 3.0),
 (140, 11, 4.0),
 (262, 497, 4.0),
 (492, 1363, 4.0)]

The train_set, is a different object...
The train_set is not a pandas dataframe, and it is not a list. It is a highly optimized C-level structure (a Cython object) designed specifically for matrix operations.

When Surprise creates the Trainset, it does something crucial: It translates your raw IDs into `"Inner IDs"`.If your pandas dataframe had a User ID of "User_99" and an Item ID of "Movie_ABC", Surprise maps them to dense, continuous integers starting at 0 (e.g., Inner User 0, Inner Item 0). This is required because matrix factorization involves creating an $N \times M$ matrix, and you can't have string indices or gaps in your matrix coordinates.

In [14]:
# 1. Look at the basic dimensions
print(f"Total Users: {train_set.n_users}")
print(f"Total Items: {train_set.n_items}")
print(f"Total Ratings: {train_set.n_ratings}")

# 2. See the mapping dictionaries (Raw ID -> Inner ID)
# Let's peek at the first 5 User mappings
user_mapping = list(train_set._raw2inner_id_users.items())[:5]
print(f"\nFirst 5 User mappings (Raw -> Inner): {user_mapping}")

# 3. Look at the actual ratings
# The ratings are stored as a generator of tuples: (inner_uid, inner_iid, rating)
# Let's pull the first 5 ratings from the generator
ratings_generator = train_set.all_ratings()
first_5_ratings = [next(ratings_generator) for _ in range(5)]
print(f"\nFirst 5 internal ratings (Inner UID, Inner IID, Rating): {first_5_ratings}")

Total Users: 610
Total Items: 8731
Total Ratings: 75627

First 5 User mappings (Raw -> Inner): [(432, 0), (288, 1), (599, 2), (42, 3), (75, 4)]

First 5 internal ratings (Inner UID, Inner IID, Rating): [(0, 0, 4.5), (0, 398, 3.0), (0, 572, 4.0), (0, 501, 2.5), (0, 1348, 3.5)]


**Creating and Training the SVD model**

In [15]:
model = SVD()
model.fit(train_set)

Under the hood, `model.fit(train_set)` has just done gradient descent and created latent embeddings for all the users and movies in the dataset.

**Testing and Evaluation**

Now that the model is trained, I need to know if it's actually any good, testing with the 25% saved data.

I will generate predictions for all the user-item pairs in the test_set and then calculate the Root Mean Square Error (RMSE).

In [16]:
predictions = model.test(test_set)

In [17]:
RMSE = rmse(predictions, verbose=True)

RMSE: 0.8804


**Exercise:**

Find the top 5 movie recommendtaions for userID 1, these must be movies the user has not yet seen and rated.

In [18]:
# get a list of movieIDs not seen by userID 1

x = rating_df[rating_df['userID'] != 1]['itemID'].to_list()

# Convert to a set, to remove duplicates
x = list(set(x))

print(len(x))

9723


In [19]:
# Make this a dataframe instead of iterating

k = pd.DataFrame(x, columns=['movieID'])
k.head()

,movieID
0,1
1,2
2,3
3,4
4,5


Create a function that we can apply to get the predicted ratings

In [20]:
def predict_ratings(movieID, user=1, model=model):
    pred = model.predict(user, movieID)
    return round(float(pred.est), 2)
    

In [21]:
# Applying the ratings to each movie prediction

k['ratings'] = k.apply(lambda x: predict_ratings(x['movieID']), axis=1)

In [22]:
k.head()

,movieID,ratings
0,1,4.53
1,2,4.28
2,3,3.95
3,4,3.83
4,5,3.72


Now to find the top 10 most recommended movies for User 1...


In [23]:
top_movies = k.sort_values(by='ratings', ascending=False)
top_movies = top_movies.head(10)

In [24]:
top_movies = list(top_movies.movieID)
top_movies

[50, 32, 2019, 475, 1089, 527, 2160, 57669, 1719, 34405]

In [25]:
movie_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [26]:
top_movies = movie_df[movie_df['movieId'].isin(top_movies)]
top_movies

,movieId,title,genres
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
46,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
413,475,In the Name of the Father (1993),Drama
461,527,Schindler's List (1993),Drama|War
828,1089,Reservoir Dogs (1992),Crime|Mystery|Thriller
1290,1719,"Sweet Hereafter, The (1997)",Drama
1494,2019,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama
1616,2160,Rosemary's Baby (1968),Drama|Horror|Thriller
5954,34405,Serenity (2005),Action|Adventure|Sci-Fi
6676,57669,In Bruges (2008),Comedy|Crime|Drama|Thriller
